In [1]:
# Extract_All_Graves_Jewish_Cemeteries.py
# 9/17/2025
# Extract All Jewish Graves from Jewish Cemetery Lisy

# fileInput="Jewish_Cemetery_List_X"
# fileOutput="Jewish_Cemeteries_All_Graves"


# fileIn=DirMain+"0-Master Worksheets\\Cemeteries\\Inputs\\"+fileInput+".csv"
# fileOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+".csv"
# fileOutInt=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+"_Interim.csv"

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import random
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date):
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out
def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
       
def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S")
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index

def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
        print('\n')
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header)
    status_code=response.status_code
    url_source = response.text
   
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser')
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text)
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header)
    status_code=response.status_code
    url_source = response.text       
    if status_code!=200:
        iTry=0
        while iTry <=4:
            print('\n** Requests Are Too Fast *** , Try=',iTry,'\n')
            time.sleep(60)
            iTry+=1
            response = requests.request("GET", url, headers=my_header)
            status_code=response.status_code
            url_source = response.text 
            if status_code==200:break
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser')
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text)
    try:
        grave_count=capture_json['total']
    except:
        specific_tag_text
        print(capture_json)
        print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
        time.sleep(60)
    return capture_json,grave_count

def trim_dataframe(dataframe_df,column):
    drop_list=['AME','Anglican','antioch','Apos','Asylum','Baptist','Bishop','Calvary','Campo',\
               'Canaan','Cathedral','Catholic','Cementerio','Cemit','Centenary','Chapel','Christ',\
               'Church','Cimet','Cimiter','Cintor','CME','Cmentarz','Colored','County','Deutscher',\
               'Dominican','Ebenezer','Friedhof','Gospel','Institute','Knight','Lutheran',\
               'Masonic ','Mennonite ','Methodist','Military','Missionary','Parish ','Presbyterian',\
               'prison','Protestant','public','Resurrection','Saint','St.','State','Township','Veterans']
    for word in drop_list:
        dataframe_df.drop(dataframe_df[dataframe_df[column].str.contains(word, case=False, na=False)].index, inplace=True) 
    dataframe_df.reset_index(drop=True, inplace = True)   
    return dataframe_df

   
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic1
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def next_search_year(search_year,interval,grave_count):
    global interval_max
    next_interval=min(int(interval*10000/grave_count),interval_max)
    first_search_year=search_year-interval
    next_search_year=first_search_year+next_interval
    last_search_year=next_search_year+next_interval
    return next_search_year,first_search_year,last_search_year,next_interval


#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++
blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract_All_Graves_Jewish_Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++

# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInput="Jewish_Cemetery_List_X"
fileOutput="Jewish_Cemeteries_All_Graves"


fileIn=DirMain+"0-Master Worksheets\\Cemeteries\\Inputs\\"+fileInput+".csv"
fileOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+".csv"
fileOutInt=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+"_Interim.csv"

if append==1:fileIn='FAG_Cemetery_Verify_List'

# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


# *************************** Read Input File and Process File +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
cemetery_df=pd.read_csv(fileIn)



# field_list=['jewish_name_count','christian_name_count','prob_jewish','prob_jewish_trunc','low_grave_count','grave_count','cemetery_name','cemetery_state','cemetery_city','cemetery_id']
# field_list_set=set(field_list)
field_list=['lastName','firstName','middleName','fullName','nickName','hasPlot','plot','birthDate','birthMonth','birthYear','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links', 'intermentHasPhoto','Parents','Spouses','Siblings','Children']
field_list_set=set(field_list)


# *************************** Read Input File and Process File +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

url_cemetery = "https://www.findagrave.com/cemetery/"
url_skip ="/memorial-search?ajax=true&skip="
url_birth_year="&limit=100&firstname=&middlename=&lastname=&birthyear="
url_interval="&birthyearfilter="
url_suffix="&lastname=&orderby=b-"

sleep_range=[8,15] 
sleep_time_pause=1*60
pause_count=100
get_counter=0

graves_df=pd.DataFrame([])
affilation_df=pd.DataFrame([])
cemetery_count=0
total_cemetery_count=len(cemetery_df)
start_year=1850
end_year=1875
first_search_year=start_year
search_year=int((end_year+start_year)/2)
last_search_year=end_year
interval=int((end_year-start_year)/2)
interval_max=25
interval_min=1

for index, row in cemetery_df.iterrows():
    first_search_year=start_year
    search_year=int((end_year+start_year)/2)
    last_search_year=end_year
    interval=int((end_year-start_year)/2)    
    cemetery_count+=1
    cemetery_id=row['cemetery_id'] 
    cemetery_name=row['cemetery_name']
    prob_jewish=row['prob_jewish']
    cemetery_grave_count=row['grave_count']
    grave_count_collected=row['graves_collected']
    grave_percent_collected=row['percent']
    
    while last_search_year<=end_year:
        frame_count=0
        grave_frame_count=0
        extracted_count=0
        grave_count=100

        print(f" $$ {cemetery_name} • {cemetery_id} • cemetery_grave_count: {cemetery_grave_count} •• {first_search_year}-{search_year}-{last_search_year} • interval:  {interval} • collected: {grave_count_collected}/{grave_percent_collected}%")
        
        while grave_frame_count<grave_count and grave_frame_count<=10000:
            # print('grave_frame_count,grave_count,interval,search_year:',grave_frame_count,grave_count,interval,search_year)
            url=url_cemetery+str(cemetery_id)+url_skip+str(grave_frame_count)+url_birth_year+str(search_year)+url_interval+str(interval)+url_suffix
            capture_json,grave_count=get_grave_count(url) 
            if status_code!=200:print('\tstatus_code:',status_code) 
            get_counter=random_sleep_new(get_counter)
            while grave_count>10000 and interval>0 :
                search_year,first_search_year,last_search_year,interval=next_search_year(search_year,interval,grave_count)
                url=url_cemetery+str(cemetery_id)+url_skip+str(grave_frame_count)+url_birth_year+str(search_year)+url_interval+str(interval)+url_suffix
                capture_json,grave_count=get_grave_count(url) 
                if status_code!=200:print('\tstatus_code:',status_code) 
                time_now_obj=datetime.datetime.now()
                time_now=time_now_obj.strftime('%H:%M:%S')
                print(f"       •••  {cemetery_name} • {cemetery_id} • grave_count: {grave_count} • {first_search_year}-{search_year}-{last_search_year} • interval:  {interval} • • Time: {time_now}") 
                get_counter=random_sleep_new(get_counter)                       
            grave_frame_df=extract_grave_data(capture_json)  
            grave_frame_df['prob_jewish']=prob_jewish  
            grave_frame_df['grave_count']=cemetery_grave_count
            grave_frame_df=clean_dataframe(grave_frame_df)
            graves_df = pd.concat([graves_df,grave_frame_df]) 
            extracted_count+=len(grave_frame_df)            
    
            time_now_obj=datetime.datetime.now()
            time_now=time_now_obj.strftime('%H:%M:%S')         
            print(f" {blank_line}", end='\r')
            print(f"   ***** {cemetery_name} • {cemetery_id} • {first_search_year}-{search_year}-{last_search_year} • interval: {interval} •• Grave Count: •  {extracted_count}/{grave_count} " +\
                  f"• • Time: {time_now} ", end='\r')        
            grave_frame_count+=100
    
            graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        # print('\n\n')
        # print('\nBefore: interval,first_search_year,search_year,last_search_year:',interval,first_search_year,search_year,last_search_year)
        if (end_year-last_search_year)==0:break
        interval0=int((end_year-last_search_year)/2)
        interval1=interval
        interval=min(interval0,interval1)
        search_year=last_search_year+interval
        if interval==0 and (end_year-search_year)>0:search_year+=1
        first_search_year=search_year-interval
        last_search_year=search_year+interval
        # print('After: interval0,interval1,interval,first_search_year,search_year,last_search_year:',first_search_year,interval0,interval1,interval,search_year,last_search_year) 
        
    if len(grave_frame_df)==0:continue
    time_now_obj=datetime.datetime.now()
    time_now=time_now_obj.strftime('%H:%M:%S') 
    print(f"\n $$ {cemetery_name} * • {cemetery_id} •• $$$$ Cemetery Count: {cemetery_count}/{total_cemetery_count} $$$$ • Total Grave Count: {grave_count}  • Time: {time_now}",'\n') 
        
    if len(graves_df)>=100000:
        def timestamp():
            today=str(datetime.date.today()).split('-')
            year=today[0]
            month=today[1]
            day=today[2]
            a=str(datetime.datetime.now())
            time_split=re.split(':| ',a)
            hour=time_split[1]
            timestamp=year+month+day+hour
            return timestamp
        timestamp_int=timestamp()
        fileOutIntT=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+"_"+timestamp_int+"_Interim.csv"
        graves_df.to_csv(fileOutIntT,mode="w", index=False,header=True)
        graves_df=pd.DataFrame([])

# get_counter=random_sleep_new(get_counter)
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")  

graves_df.to_csv(fileOut,mode="w", index=False,header=True)
# graves_df.to_csv(fileOut0,mode="w", index=False,header=True)
print("\n")
print('Uploaded To: ',fileOut)


**Starting ***  Extract_All_Graves_Jewish_Cemeteries   *** , Time: 2025-11-03 12:32:23.864909  **

 $$ Adath Israel Cemetery • 2300526 • cemetery_grave_count: 838 •• 1850-1862-1875 • interval:  12 • collected: 43.0/5.10%%
   ***** Adath Israel Cemetery • 2300526 • 1850-1862-1875 • interval: 12 •• Grave Count: •  79/79 • • Time: 12:32:35                                                                        
 $$ Adath Israel Cemetery * • 2300526 •• $$$$ Cemetery Count: 1/1541 $$$$ • Total Grave Count: 79  • Time: 12:32:35 

 $$ Adath Israel Cemetery • 2370933 • cemetery_grave_count: 86 •• 1850-1862-1875 • interval:  12 • collected: 14.0/16.30%%
   ***** Adath Israel Cemetery • 2370933 • 1850-1862-1875 • interval: 12 •• Grave Count: •  1/1 • • Time: 12:32:46                                                                          
 $$ Adath Israel Cemetery * • 2370933 •• $$$$ Cemetery Count: 2/1541 $$$$ • Total Grave Count: 1  • Time: 12:32:46 

 $$ Adath Israel Cemetery • 2504523 • ceme

In [41]:
# interval= 7 
# search_year=1913 
# first_search_year=1875 
# last_search_year=1920


if interval==0 and (end_year-search_year)==0:break
interval0=int((end_year-last_search_year)/2)
interval1=interval
interval=min(interval0,interval1)
search_year=last_search_year+interval
if interval==0 and (end_year-search_year)>0:search_year+=1
first_search_year=search_year-interval
last_search_year=search_year+interval
print('After: interval0,interval1,interval,search_year,first_search_year,last_search_year:',interval0,interval1,interval,search_year,first_search_year,last_search_year)


After: interval0,interval1,interval,search_year,first_search_year,last_search_year: 0 2 0 1925 1925 1925
